# Qwen3.5-9B LoRA Fine-Tuning — Computer Use

## Overview

This notebook fine-tunes a **LoRA adapter** on top of Qwen3.5-9B using a synthetically generated computer-use dataset. The result is a tiny adapter file (~110 MB) that, when attached to the base model, gives it native tool-use and computer-control capabilities.

## Training strategy

- **LoRA (Low-Rank Adaptation)** — only rank-16 delta matrices are trained; base weights stay frozen.
- **Unsloth** — CUDA-kernel-patched model loader with fused RoPE, FlashAttention, and smart gradient checkpointing to minimise VRAM.
- **bfloat16 throughout** — no 4-bit quantisation during training so the adapter can be merged cleanly into fp16 later.
- **Multi-turn `CompletionOnlyCollator`** — computes loss only on assistant turns, maximising signal from multi-step tool-use conversations.

## Outputs

Adapter checkpoint (`OUTPUT_DIR`) containing:
- `adapter_config.json` — LoRA hyperparameters + base model reference
- `adapter_model.safetensors` — trained LoRA weight matrices
- Tokenizer files

> **After training**, use the `[Checkpoint_Resume]` notebook to merge the adapter into the base model and export to GGUF.

## Cell 1 — Imports

All required libraries. `time` is used to measure training duration; `SFTTrainer` / `SFTConfig` are TRL's supervised fine-tuning wrappers that integrate directly with Unsloth.

In [ ]:
from unsloth import FastLanguageModel
import os
import time
import json
import shutil
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import PeftModel

print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
print("Unsloth + TRL import OK 🦥")

## Cell 2 — Configuration

**Edit these variables before running.** All subsequent cells derive from them.

| Variable | Description |
|---|---|
| `MODEL_NAME` | HuggingFace model ID for the Qwen3.5-9B base (loaded from Hub or local cache) |
| `DATASET_PATH` | Path to the JSONL file containing synthetic computer-use conversations |
| `OUTPUT_DIR` | Directory where adapter checkpoints will be saved during and after training |
| `MAX_SEQ_LENGTH` | Maximum token length per example — 4096 is a good balance for tool-use conversations |

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────
# Priority: Env var (OI_MCP_PATH) > Current directory discovery
def get_base_path():
    path = os.environ.get("OI_MCP_PATH")
    if path:
        return path.strip('"\'')
    # Default to current repo root if no env var is found
    return os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

BASE_DIR = get_base_path()

MODEL_NAME            = "Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled"
MODEL_PATH            = r"C:\Users\Rhushabh\.cache\huggingface\hub\models--Jackrong--Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\snapshots\59f57e471e041fe27ee3f98dba2ec02a50817afc"
DATASET_PATH          = os.path.join(BASE_DIR, "computer-use-finetuning", "datasets", "synthetic_dataset.jsonl")
ADAPTER_CHECKPOINT    = os.path.join(BASE_DIR, "computer-use-finetuning", "models", "adapter-checkpoint")
OUTPUT_DIR            = os.path.join(BASE_DIR, "computer-use-finetuning", "models", "qwen-tool-use-lora-adapter-qwen-27b-gguf")

# ── Sequence length ────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048

print(f"Base dir   : {BASE_DIR}")
print(f"Model      : {MODEL_NAME}")
print(f"Model dir     : {MODEL_PATH}")
print(f"Dataset    : {DATASET_PATH}")
print(f"Adapter Chekpoint dir    : {ADAPTER_CHECKPOINT}")
print(f"Output dir : {OUTPUT_DIR}")

## Cell 3 — Load Model & Tokenizer

Load Qwen3.5-9B via Unsloth's optimised loader. Unsloth automatically applies:
- Fused RoPE embeddings (faster + less memory)
- FlashAttention-2 (if supported)
- Smart gradient checkpointing ready for the LoRA step

`load_in_4bit=True` is used during training to keep VRAM within 16 GB; the 4-bit quantisation is applied only to the frozen base weights and does **not** affect training precision (LoRA layers train in bf16). `trust_remote_code=True` is required for Qwen's custom tokenizer code.

In [ ]:
# Load the base model + tokenizer using Unsloth's kernel-patched loader.
# Unsloth automatically applies fused RoPE, FlashAttention, and other speed-ups.
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_PATH,
    max_seq_length     = MAX_SEQ_LENGTH,
    dtype              = torch.bfloat16,  # bf16 for training; keeps numerical range stable
    load_in_4bit       = True,            # 4-bit quant on frozen base weights → saves ~8 GB VRAM
    trust_remote_code  = True,            # required for Qwen's custom tokenizer
    device_map         = {"": 0},          # force everything on GPU 0
    offload_folder     = None,
)
print(f"Model loaded: {MODEL_NAME}")

## Cell 4 — Dataset Preparation

Load the JSONL dataset and apply Qwen's native chat template to convert each `messages` list into a single formatted string.

- `apply_chat_template` renders system/user/assistant turns into `<|im_start|>...<|im_end|>` format that matches Qwen's pre-training.
- `add_generation_prompt=False` ensures the trailing `<|im_start|>assistant\n` prompt is **not** added — we want the full turn including the assistant response so the model learns to produce it.
- The resulting `text` field is what the tokenizer will later consume.

In [ ]:
print(f"Loading dataset from {DATASET_PATH}...")
# Load all examples from the JSONL file as a single training split
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Dataset size: {len(dataset)} examples")

def format_chat_template(example):
    """Convert 'messages' list to a single chat-formatted string using Qwen's template.
    Each turn becomes: <|im_start|>role\ncontent<|im_end|>
    add_generation_prompt=False includes the full assistant turn (including response)
    so the model learns to generate it.
    """
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize            = False,
        add_generation_prompt = False,  # include full assistant response, not just the prompt
    )
    return example

dataset = dataset.map(format_chat_template)
print(f"Chat template applied. Sample:\n{dataset[0]['text'][:300]}...")

## Cell 5 — LoRA Adapter Configuration

Wrap the frozen base model with trainable LoRA rank-16 matrices using Unsloth's `get_peft_model`.

**Why these target modules?**
- `q_proj`, `k_proj`, `v_proj`, `o_proj` — attention projections; fine-tuning these teaches the model *what to attend to* in tool-use contexts.
- `gate_proj`, `up_proj`, `down_proj` — FFN projections; fine-tuning these teaches the model *what to say* (the actual tool-call JSON, reasoning, etc.).

**LoRA hyperparameters:**

| Param | Value | Reason |
|---|---|---|
| `r` | 16 | Good capacity for a domain-specific task; higher r = more trainable params |
| `lora_alpha` | 16 | `alpha/r = 1.0` effective scale — no extra amplification |
| `lora_dropout` | 0 | Disabled; Unsloth has special optimisations for `dropout=0` |
| `bias` | none | Disabled; Unsloth has special optimisations for `bias=none` |

In [ ]:
print("Attaching LoRA adapter to the base model...")

# Wrap the frozen base model with trainable LoRA matrices.
# Unsloth's get_peft_model builds its own optimised LoRA graph (not vanilla PEFT),
# giving access to save_pretrained_merged and save_pretrained_gguf later.
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 32,                 # Increased from 16 for better tool-use capacity
    target_modules             = [                   # Attention + FFN projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha                 = 32,                 # Increased from 16
    lora_dropout               = 0,                  # Unsloth has special kernels for dropout=0
    bias                       = "none",             # Unsloth has special kernels for bias=none
    use_gradient_checkpointing = "unsloth",          # Unsloth's smart checkpointing — saves ~40% VRAM
    random_state               = 3407,               # Reproducibility seed
)
print(model.print_trainable_parameters())

## Cell 6 — Trainer Setup (Collator + Tokenization + SFTTrainer)

This is the core training setup cell. It handles three things:

### 6a. EOS/PAD token synchronisation
Qwen's tokenizer wraps an inner tokenizer inside a `Processor`. Both must have matching EOS and PAD tokens to avoid training on incorrect end-of-sequence boundaries. `<|im_end|>` is Qwen's native turn delimiter and is the correct EOS for this format.

### 6b. `CompletionOnlyCollator`
A custom data collator that:
- Sets all token labels to `-100` (masked, no gradient) by default.
- Scans each sequence for `<|im_start|>assistant\n` markers.
- Unmasks **every** assistant turn found — not just the first one.

This is critical for multi-step tool-use conversations where the model must produce multiple assistant turns (reasoning → tool call → observe result → final answer). A single-turn collator would lose 80%+ of training signal.

### 6c. Pre-tokenization
Tokenizes the full dataset in the main process before training. This avoids a known pickling crash in DataLoader worker processes when using Unsloth's wrapped tokenizer.

### 6d. `SFTTrainer` + `SFTConfig`
Key hyperparameters:

| Param | Value | Reason |
|---|---|---|
| `per_device_train_batch_size` | 2 | Fits in 16 GB VRAM with bf16 + gradient checkpointing |
| `gradient_accumulation_steps` | 4 | Effective batch = 2×4 = 8 — stable gradients |
| `warmup_steps` | 50 | ~5% of 900 steps — prevents early instability |
| `max_steps` | 900 | ~5 epochs on the dataset |
| `learning_rate` | 2e-4 | Standard LoRA LR; cosine decay from here |
| `save_steps` | 100 | Checkpoint every 100 steps |
| `lr_scheduler_type` | cosine | Smooth LR decay towards end of training |

In [ ]:
# ---------------------------------------------------------------------------
# TRAINER SETUP v3 — Multi-turn collator + pre-tokenization
# ---------------------------------------------------------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. Find correct EOS from inner tokenizer
inner_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
real_eos_token = "<|im_end|>"
for token in ["<|im_end|>", "<|endoftext|>", "</s>"]:
    try:
        if token in inner_tokenizer.get_vocab():
            real_eos_token = token
            break
    except Exception:
        continue
real_eos_id = inner_tokenizer.convert_tokens_to_ids(real_eos_token)
print(f"EOS token: '{real_eos_token}' (id={real_eos_id})")

# 2. Forced Synchronization
for obj in [tokenizer, inner_tokenizer, model.config]:
    setattr(obj, "eos_token",    real_eos_token)
    setattr(obj, "eos_token_id", real_eos_id)
    setattr(obj, "pad_token",    real_eos_token)
    setattr(obj, "pad_token_id", real_eos_id)
if hasattr(inner_tokenizer, "add_special_tokens"):
    inner_tokenizer.add_special_tokens({"eos_token": real_eos_token, "pad_token": real_eos_token})

# 3. MULTI-TURN CompletionOnlyCollator
IM_START_ID = 248045  # <|im_start|> token id for Qwen

class CompletionOnlyCollator:
    """Masks all prompt/system/user tokens. Trains on ALL assistant turns."""
    def __init__(self, response_template, tokenizer):
        self.tokenizer = tokenizer
        self.response_template_ids = tokenizer.encode(
            response_template, add_special_tokens=False
        )

    def __call__(self, features):
        batch = self.tokenizer.pad(features, return_tensors="pt", padding=True)
        labels = batch["input_ids"].clone()
        labels[:] = -100  # mask everything by default

        tpl = self.response_template_ids
        n = len(tpl)

        for i in range(len(labels)):
            ids = batch["input_ids"][i].tolist()
            j = 0
            while j <= len(ids) - n:
                if ids[j:j + n] == tpl:
                    response_start = j + n
                    response_end = len(ids)
                    for k in range(response_start, len(ids)):
                        if ids[k] == IM_START_ID:
                            response_end = k
                            break
                    labels[i][response_start:response_end] = \
                        batch["input_ids"][i][response_start:response_end]
                    j = response_end
                else:
                    j = j + 1

        batch["labels"] = labels
        return batch

response_template = "<|im_start|>assistant\n"
collator = CompletionOnlyCollator(response_template, inner_tokenizer)

# 4. Pre-tokenize dataset in main process (fixes pickle crash)
def tokenize_example(example):
    encoded = inner_tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        return_attention_mask=True,
    )
    return {"input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"]}

print(f"Pre-tokenizing {len(dataset)} examples...")
tokenized_dataset = dataset.map(
    tokenize_example,
    num_proc=1,
    remove_columns=[c for c in dataset.column_names if c not in ("input_ids", "attention_mask")],
    desc="Tokenizing",
)

# 5. Training Config
training_args = SFTConfig(
    output_dir=ADAPTER_CHECKPOINT,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,           # Increased from 2e-5 for faster convergence on schema
    logging_steps=10,
    max_steps=1600,               # Increased from 500 to cover 6650 records ~2x
    warmup_steps=100,             # Added warmup for stability
    save_steps=200,               # Adjusted for longer run
    optim="adamw_8bit",
    fp16=False,
    bf16=True,
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1,
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    processing_class=inner_tokenizer,
    args=training_args,
    data_collator=collator,
)

print(f"Trainer EOS Token determined as: {real_eos_token} (ID: {real_eos_id})")

## Cell 7 — Training Execution + Metrics + Adapter Save

This cell fires the actual training run and handles three phases:

### 7a. Pre-flight GPU snapshot
Captures current VRAM reservation and total GPU memory before training begins. Printed at the end for comparison.

### 7b. `trainer.train()`
Launches the SFTTrainer loop. Unsloth's `ACCELERATE_BYPASS_DEVICE_MAP=true` environment flag is required to avoid a device-map conflict when using its custom model wrapper.

### 7c. Post-training metrics
After training completes:
- **Total Training Time** — wall-clock minutes for the full run
- **Peak VRAM Usage** — maximum GPU memory reserved during training (GB)
- **Steps/Sec** — throughput sampled from `train_samples_per_second`

### 7d. Adapter save
Only the **LoRA delta weights** are saved — not the full 9B base model. This produces a tiny `~110 MB` adapter directory containing:
- `adapter_config.json` — LoRA rank, alpha, target modules, base model reference
- `adapter_model.safetensors` — the trained rank-16 weight matrices
- Tokenizer files (for standalone inference)

> **Note:** To use the adapter you must load the base model first and attach the adapter via `PeftModel.from_pretrained()` or Unsloth's `from_pretrained()` with `adapter_name`.

In [ ]:
# ---------------------------------------------------------------------------
# TRAINING EXECUTON
# ---------------------------------------------------------------------------
print("Starting ultra-fast Unsloth training...")
start_time = time.time()
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"
trainer_stats = trainer.train()

end_time = time.time()
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n--- TRAINING METRICS ---")
print(f"Total Training Time: {(end_time - start_time)/60:.2f} minutes")
print(f"Peak VRAM Usage:     {used_memory} GB")
print(f"Steps/Sec:           {trainer_stats.metrics['train_samples_per_second']:.2f}")

# Save ONLY the adapters
print(f"Finished! Saving LoRA adapters to {ADAPTER_CHECKPOINT}")
model.save_pretrained(ADAPTER_CHECKPOINT) # Local saving
tokenizer.save_pretrained(ADAPTER_CHECKPOINT)
print(f"✅ Adapter successfully saved!")

## Cell 8 — Export to GGUF (Optional)

Converts the merged adapter + base model into a standalone **GGUF** file for hardware-agnostic local inference. GGUF is the native format for `llama.cpp` and compatible runtimes (Ollama, LM Studio, koboldcpp, etc.).

### What `save_pretrained_gguf` does
1. **Merges** the LoRA adapter back into the base model weights (fp16)
2. **Quantises** the merged model using the specified method
3. **Writes** a single `.gguf` file to the `"model"` directory

### Quantisation method: `q4_k_m`
| Method | Bits | Size (9B) | Quality vs Size |
|--------|------|-----------|------------------|
| `q4_k_m` | 4-bit | ~5.7 GB | Best balance — recommended for most hardware |
| `q8_0` | 8-bit | ~9.4 GB | Near lossless, requires more RAM |
| `f16` | 16-bit | ~17 GB | Full precision, for fine-tuning or re-export |

> **Prerequisite:** This cell requires that the adapter was saved in Cell 7 and that the base model weights are accessible on disk. The merge step loads the full fp16 base model (~18 GB) into RAM temporarily.

> **Tip:** Skip this cell if you only need the raw adapter for server-side inference with VLLM or TGI — those load the adapter directly without needing a GGUF merge.

In [ ]:
# Reload base + attach LoRA adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = MODEL_NAME,
    max_seq_length    = MAX_SEQ_LENGTH,
    dtype             = torch.bfloat16,
    load_in_4bit      = True,          # same as training
    trust_remote_code = True,
)

model = PeftModel.from_pretrained(model, ADAPTER_CHECKPOINT)

quants = ["q2_k", "q3_k_s", "q3_k_m"]  # multiple versions

for quant in quants:
    print(f"Exporting quantization: {quant}")
    GGUF_OUTPUT_DIR = os.path.join(OUTPUT_DIR, f"qwen35-27b-tooluse-{quant}")
    model.save_pretrained_gguf(
        GGUF_OUTPUT_DIR,
        f"qwen35-27b-tooluse-{quant}",
        tokenizer,
        quantization_method  = quant,
        maximum_memory_usage = 0.85,   # ~5–6GB of your 16GB VRAM, rest uses 32GB RAM + disk
    )
    torch.cuda.empty_cache()
    print(f"Done: {quant}")
